In [ ]:
from IPython.display import HTML

from autosim.experimental.simulations import ReactionDiffusion
from autosim.utils import plot_spatiotemporal_video

## Dataset grid comparison

Load saved datasets from disk and display a snapshot from each in a tiled grid.
The bottom row spans the full width at half height for the 4:1 aspect-ratio dataset.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from matplotlib.gridspec import GridSpec

datasets = [
    ("reaction_diffusion_e3e8515", "Reaction-Diffusion", 0, -1),
    ("gray_scott_68b0669", "Gray-Scott", 0, -1),
    ("advection_diffusion_2ba25b9", "Advection-Diffusion", 0, -1),
    ("conditioned_navier_stokes_2d_5e1f575", "Conditioned Navier-Stokes", 1, 100),
    ("shallow_water2d_9057134", "Shallow Water", 1, 100),
    ("gpe/laser_only_wake_9be0bfb", "GPE", 0, -1),
    ("lattice_boltzmann_f530ec5", "Lattice Boltzmann", 1, -1),
]

# Rotate colormaps to make each panel visually distinct.
cmaps = ["viridis", "magma", "cividis", "plasma", "RdBu", "turbo", "inferno"]

base_dir = Path("generated_datasets")
batch_idx, time_idx, channel = 0, -1, 0  # which snapshot to show

# --- build grid: 3 rows x 2 cols + 1 bottom row ---
# figsize chosen so two square columns can sit flush without center gap.
fig = plt.figure(figsize=(6, 12))
fig.patch.set_alpha(0.0)
gs = GridSpec(4, 2, height_ratios=[1, 1, 1, 1], hspace=0.0, wspace=0.0)
base_dir = Path(os.getenv("AUTOCAST_DATASETS"))  # type: ignore  # noqa: PGH003

panel_width = None
panel_height = None

# first 6 datasets in 3x2
for i, (rel_path, _, channel, time_idx) in enumerate(datasets[:6]):
    row, col = divmod(i, 2)
    ax = fig.add_subplot(gs[row, col])
    ax.patch.set_alpha(0.0)
    path = base_dir / rel_path / "train" / "data.pt"
    if path.exists():
        data = torch.load(path, weights_only=False)["data"]
        ax.imshow(
            data[batch_idx, time_idx, :, :, channel].numpy(),
            cmap=cmaps[i % len(cmaps)],
            aspect="equal",
        )
    ax.axis("off")

    if panel_width is None or panel_height is None:
        bbox = ax.get_position()
        panel_width = bbox.width
        panel_height = bbox.height

# 7th dataset — square crop and centered bottom placement with panel-matched size.
ax_lattice = fig.add_subplot(gs[3, :])
ax_lattice.patch.set_alpha(0.0)
bottom_bbox = ax_lattice.get_position()
if panel_width is not None and panel_height is not None:
    ax_lattice.set_position(
        [0.5 - panel_width / 2, bottom_bbox.y0, panel_width, panel_height]
    )

rel_path, _, channel, time_idx = datasets[6]
path = base_dir / rel_path / "train" / "data.pt"
if path.exists():
    data = torch.load(path, weights_only=False)["data"]
    lattice_frame = data[batch_idx, time_idx, :, :, channel].numpy()
    h, w = lattice_frame.shape

    if w > h:
        x0 = (w - h) // 2
        lattice_frame = lattice_frame[:, x0 : x0 + h]
    elif h > w:
        y0 = (h - w) // 2
        lattice_frame = lattice_frame[y0 : y0 + w, :]

    ax_lattice.imshow(
        lattice_frame,
        cmap=cmaps[6 % len(cmaps)],
        aspect="equal",
    )
ax_lattice.axis("off")

fig.subplots_adjust(left=0.0, right=1.0, bottom=0.0, top=1.0, wspace=0.0, hspace=0.0)
plt.show()

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from matplotlib.gridspec import GridSpec

datasets = [
    ("reaction_diffusion_e3e8515", "Reaction-Diffusion", 0, -1),
    ("gray_scott_68b0669", "Gray-Scott", 0, -1),
    ("advection_diffusion_2ba25b9", "Advection-Diffusion", 0, -1),
    ("conditioned_navier_stokes_2d_5e1f575", "Conditioned Navier-Stokes", 1, 100),
    ("shallow_water2d_9057134", "Shallow Water", 1, 100),
    ("gpe/laser_only_wake_9be0bfb", "GPE", 0, -1),
    # ("lattice_boltzmann_f530ec5", "Lattice Boltzmann", 1, -1),
]

# Rotate colormaps to make each panel visually distinct.
cmaps = ["viridis", "magma", "cividis", "plasma", "RdBu", "turbo", "inferno"]

base_dir = Path("generated_datasets")
batch_idx, time_idx, channel = 0, -1, 0  # which snapshot to show

# --- build grid: 3 rows x 2 cols + 1 bottom row ---
# figsize chosen so two square columns can sit flush without center gap.
fig = plt.figure(figsize=(6, 9))
fig.patch.set_alpha(0.0)
# gs = GridSpec(4, 2, height_ratios=[1, 1, 1, 1], hspace=0.0, wspace=0.0)
gs = GridSpec(3, 2, height_ratios=[1, 1, 1], hspace=0.0, wspace=0.0)
base_dir = Path(os.getenv("AUTOCAST_DATASETS"))  # type: ignore  # noqa: PGH003

panel_width = None
panel_height = None

# first 6 datasets in 3x2
for i, (rel_path, _, channel, time_idx) in enumerate(datasets[:6]):
    row, col = divmod(i, 2)
    ax = fig.add_subplot(gs[row, col])
    ax.patch.set_alpha(0.0)
    path = base_dir / rel_path / "train" / "data.pt"
    if path.exists():
        data = torch.load(path, weights_only=False)["data"]
        ax.imshow(
            data[batch_idx, time_idx, :, :, channel].numpy(),
            cmap=cmaps[i % len(cmaps)],
            aspect="equal",
        )
    ax.axis("off")

    if panel_width is None or panel_height is None:
        bbox = ax.get_position()
        panel_width = bbox.width
        panel_height = bbox.height


fig.subplots_adjust(left=0.0, right=1.0, bottom=0.0, top=1.0, wspace=0.0, hspace=0.0)
fig.savefig("visualization.png", dpi=300, bbox_inches="tight", pad_inches=0.0)
plt.show()

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from IPython.display import HTML
from matplotlib.animation import FFMpegWriter, FuncAnimation, PillowWriter
from matplotlib.gridspec import GridSpec

# (dataset_path, label, channel)
datasets = [
    ("reaction_diffusion_e3e8515", "Reaction-Diffusion", 0),
    ("gray_scott_68b0669", "Gray-Scott", 0),
    ("advection_diffusion_2ba25b9", "Advection-Diffusion", 0),
    ("conditioned_navier_stokes_2d_5e1f575", "Conditioned Navier-Stokes", 0),
    ("shallow_water2d_9057134", "Shallow Water", 1),
    ("gpe/laser_only_wake_9be0bfb", "GPE", 0),
]

cmaps = ["viridis", "magma", "cividis", "plasma", "RdBu", "turbo", "inferno"]
base_dir = Path(os.getenv("AUTOCAST_DATASETS", "generated_datasets"))
batch_idx = 0

fig = plt.figure(figsize=(6, 9))
fig.patch.set_alpha(0.0)
gs = GridSpec(3, 2, height_ratios=[1, 1, 1], hspace=0.0, wspace=0.0)

ims = []
lengths = []

for i, (rel_path, title, channel) in enumerate(datasets[:6]):
    row, col = divmod(i, 2)
    ax = fig.add_subplot(gs[row, col])
    ax.patch.set_alpha(0.0)
    ax.axis("off")

    path = base_dir / rel_path / "train" / "data.pt"
    if path.exists():
        data = torch.load(path, weights_only=False)["data"]
        sequence = data[batch_idx, :, :, :, channel].cpu().numpy()
        lengths.append(sequence.shape[0])
        im = ax.imshow(sequence[0], cmap=cmaps[i % len(cmaps)], aspect="equal", animated=True)
        ims.append((im, sequence))
    else:
        placeholder = torch.zeros((64, 64)).numpy()
        lengths.append(1)
        im = ax.imshow(placeholder, cmap=cmaps[i % len(cmaps)], aspect="equal", animated=True)
        ax.text(0.5, 0.5, f"Missing\n{title}", ha="center", va="center", color="white", fontsize=8, transform=ax.transAxes)
        ims.append((im, placeholder[None, ...]))

fig.subplots_adjust(left=0.0, right=1.0, bottom=0.0, top=1.0, wspace=0.0, hspace=0.0)
max_frames = max(lengths) if lengths else 1


def update(frame_idx: int):
    artists = []
    for im, sequence in ims:
        t = frame_idx % sequence.shape[0]
        im.set_array(sequence[t])
        artists.append(im)
    return artists

ani = FuncAnimation(fig, update, frames=max_frames, interval=80, blit=True, repeat=True)

export_dir = Path("outputs")
export_dir.mkdir(parents=True, exist_ok=True)
gif_path = export_dir / "tiles_animation.gif"
mp4_path = export_dir / "tiles_animation.mp4"
fps = 12

ani.save(gif_path, writer=PillowWriter(fps=fps), dpi=160)
print(f"Saved GIF: {gif_path.resolve()}")

try:
    ani.save(mp4_path, writer=FFMpegWriter(fps=fps, bitrate=1800), dpi=160)
    print(f"Saved MP4: {mp4_path.resolve()}")
except (RuntimeError, FileNotFoundError) as exc:
    print(f"MP4 export skipped ({exc}). Install ffmpeg to enable MP4 output.")

plt.close(fig)
HTML(ani.to_jshtml())

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from IPython.display import HTML
from matplotlib.animation import FFMpegWriter, FuncAnimation, PillowWriter
from matplotlib.gridspec import GridSpec

# (dataset_path, label, channel)
datasets = [
    # ("reaction_diffusion_e3e8515", "Reaction-Diffusion", 0),
    # ("gray_scott_68b0669", "Gray-Scott", 0),
    ("advection_diffusion_2ba25b9", "Advection-Diffusion", 0),
    ("conditioned_navier_stokes_2d_5e1f575", "Conditioned Navier-Stokes", 0),
    ("shallow_water2d_9057134", "Shallow Water", 1),
    ("gpe/laser_only_wake_9be0bfb", "GPE", 0),
]

cmaps = ["viridis", "magma", "cividis", "plasma", "RdBu", "turbo", "inferno"][2:]
base_dir = Path(os.getenv("AUTOCAST_DATASETS", "generated_datasets"))
batch_idx = 0

fig = plt.figure(figsize=(6, 6))
fig.patch.set_alpha(0.0)
gs = GridSpec(2, 2, height_ratios=[1, 1], hspace=0.0, wspace=0.0)

ims = []
lengths = []
max_frames = 200
for i, (rel_path, title, channel) in enumerate(datasets):
    row, col = divmod(i, 2)
    ax = fig.add_subplot(gs[row, col])
    ax.patch.set_alpha(0.0)
    ax.axis("off")
    if i == 0:
        continue
    path = base_dir / rel_path / "train" / "data.pt"
    if path.exists():
        data = torch.load(path, weights_only=False)["data"]
        sequence = data[batch_idx, :max_frames, :, :, channel].cpu().numpy()
        lengths.append(sequence.shape[0])
        im = ax.imshow(sequence[0], cmap=cmaps[i % len(cmaps)], aspect="equal", animated=True)
        ims.append((im, sequence))
    else:
        placeholder = torch.zeros((64, 64)).numpy()
        lengths.append(1)
        im = ax.imshow(placeholder, cmap=cmaps[i % len(cmaps)], aspect="equal", animated=True)
        ax.text(0.5, 0.5, f"Missing\n{title}", ha="center", va="center", color="white", fontsize=8, transform=ax.transAxes)
        ims.append((im, placeholder[None, ...]))

fig.subplots_adjust(left=0.0, right=1.0, bottom=0.0, top=1.0, wspace=0.0, hspace=0.0)
max_frames = max(lengths) if lengths else 1


def update(frame_idx: int):
    artists = []
    for im, sequence in ims:
        t = frame_idx % sequence.shape[0]
        im.set_array(sequence[t])
        artists.append(im)
    return artists

ani = FuncAnimation(fig, update, frames=max_frames, interval=80, blit=True, repeat=True)

export_dir = Path("outputs")
export_dir.mkdir(parents=True, exist_ok=True)
gif_path = export_dir / "tiles_animation.gif"
mp4_path = export_dir / "tiles_animation.mp4"
fps = 12

ani.save(gif_path, writer=PillowWriter(fps=fps), dpi=160)
print(f"Saved GIF: {gif_path.resolve()}")

try:
    ani.save(mp4_path, writer=FFMpegWriter(fps=fps, bitrate=1800), dpi=160)
    print(f"Saved MP4: {mp4_path.resolve()}")
except (RuntimeError, FileNotFoundError) as exc:
    print(f"MP4 export skipped ({exc}). Install ffmpeg to enable MP4 output.")

plt.close(fig)
HTML(ani.to_jshtml())

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from IPython.display import HTML
from matplotlib.animation import FFMpegWriter, FuncAnimation, PillowWriter
from matplotlib.gridspec import GridSpec

# (dataset_path, label, channel)
datasets = [
    # ("reaction_diffusion_e3e8515", "Reaction-Diffusion", 0),
    # ("gray_scott_68b0669", "Gray-Scott", 0),
    # ("advection_diffusion_2ba25b9", "Advection-Diffusion", 0),
    # ("conditioned_navier_stokes_2d_5e1f575", "Conditioned Navier-Stokes", 0),
    # ("shallow_water2d_9057134", "Shallow Water", 1),
    # ("gpe/laser_only_wake_9be0bfb", "GPE", 0),
    ("advection_diffusion_2ba25b9", "Advection-Diffusion", 0),
    ("128x128/conditioned_navier_stokes_2d_b106e4d", "Conditioned Navier-Stokes", 0),
    ("128x128/shallow_water2d_433068d/", "Shallow Water", 1),
    ("128x128/gpe/laser_only_wake_65c1425", "GPE", 0),
]

cmaps = ["viridis", "magma", "cividis", "plasma", "RdBu", "turbo", "inferno"][2:]
base_dir = Path(os.getenv("AUTOCAST_DATASETS", "generated_datasets"))
batch_idx = 0

fig = plt.figure(figsize=(6, 6))
fig.patch.set_alpha(0.0)
gs = GridSpec(2, 2, height_ratios=[1, 1], hspace=0.0, wspace=0.0)

ims = []
lengths = []
max_frames = 200
for i, (rel_path, title, channel) in enumerate(datasets):
    row, col = divmod(i, 2)
    ax = fig.add_subplot(gs[row, col])
    ax.patch.set_alpha(0.0)
    ax.axis("off")
    if i == 0:
        continue
    path = base_dir / rel_path / "train" / "data.pt"
    print(path)
    if path.exists():
        data = torch.load(path, weights_only=False)["data"]
        sequence = data[batch_idx, :max_frames, :, :, channel].cpu().numpy()
        lengths.append(sequence.shape[0])
        im = ax.imshow(sequence[0], cmap=cmaps[i % len(cmaps)], aspect="equal", animated=True)
        ims.append((im, sequence))
    else:
        placeholder = torch.zeros((64, 64)).numpy()
        lengths.append(1)
        im = ax.imshow(placeholder, cmap=cmaps[i % len(cmaps)], aspect="equal", animated=True)
        ax.text(0.5, 0.5, f"Missing\n{title}", ha="center", va="center", color="white", fontsize=8, transform=ax.transAxes)
        ims.append((im, placeholder[None, ...]))

fig.subplots_adjust(left=0.0, right=1.0, bottom=0.0, top=1.0, wspace=0.0, hspace=0.0)
max_frames = max(lengths) if lengths else 1


def update(frame_idx: int):
    artists = []
    for im, sequence in ims:
        t = frame_idx % sequence.shape[0]
        im.set_array(sequence[t])
        artists.append(im)
    return artists

ani = FuncAnimation(fig, update, frames=max_frames, interval=80, blit=True, repeat=True)

export_dir = Path("outputs")
export_dir.mkdir(parents=True, exist_ok=True)
gif_path = export_dir / "tiles_animation.gif"
mp4_path = export_dir / "tiles_animation.mp4"
fps = 12

ani.save(gif_path, writer=PillowWriter(fps=fps), dpi=300)
print(f"Saved GIF: {gif_path.resolve()}")

try:
    ani.save(mp4_path, writer=FFMpegWriter(fps=fps, bitrate=1800), dpi=300)
    print(f"Saved MP4: {mp4_path.resolve()}")
except (RuntimeError, FileNotFoundError) as exc:
    print(f"MP4 export skipped ({exc}). Install ffmpeg to enable MP4 output.")

plt.close(fig)
HTML(ani.to_jshtml())